In [1]:
import pandas as pd
import numpy as np

# Salt hash 

In [4]:
import os
os.environ["ANON_SALT"] = "427558aafb897ee5ea62a0499d7df5b5a027ec2b3ddfc3471b49c44c61bdafa"

In [5]:
import os
print("salt loaded:", bool(os.environ.get("ANON_SALT")))

salt loaded: True


### Seperating from file Step 1 Setup and hashing with SALT

In [8]:
import os, glob, re, hashlib
import pandas as pd
from tqdm import tqdm

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")

PARTS = "data/interim/parts"
CLEAN = "data/interim/clean"           # feature-eligible subs
CLEAN_DAPT = "data/interim/clean_dapt" # popculture, DAPT-only
os.makedirs(CLEAN, exist_ok=True)
os.makedirs(CLEAN_DAPT, exist_ok=True)

SALT = os.environ.get("ANON_SALT")
assert SALT, "set ANON_SALT before running (export ANON_SALT='...')"

DAPT_ONLY = {"popculture"}   # cleaned but routed away from features

### Hashing text and normalisation cleaning up

In [10]:
import ftfy
import re
import hashlib

# Non-users to skip (bot + Reddit placeholders) -> excluded, not hashed
BOTS = {"AutoModerator", "[deleted]", "[removed]", None}

def hash_user(name):
    # Irreversible pseudonym: SHA-256 of salt+username, first 16 hex chars.
    # Same user -> same ID (for aggregation); salt blocks reversal.
    if name in BOTS:
        return None
    return hashlib.sha256((SALT + name.lower()).encode()).hexdigest()[:16]

# Precompiled once, reused per row:
URL     = re.compile(r"https?://\S+|www\.\S+")   # links
MENTION = re.compile(r"/?u/[A-Za-z0-9_-]+")      # user mentions
SUBLINK = re.compile(r"/?r/([A-Za-z0-9_]+)")     # subreddit refs

def clean_text(t):
    # Strips identity + boilerplate; KEEPS emoji/casing/slang (that's the signal)
    if not isinstance(t, str):
        return ""
    t = ftfy.fix_text(t)                  # fix broken Unicode
    t = URL.sub(" <URL> ", t)             # mask links
    t = MENTION.sub(" <USER> ", t)        # mask mentions (identity)
    t = SUBLINK.sub(r" r/\1 ", t)         # keep sub names (topical)
    t = t.replace("&amp;", "&").replace("&gt;", ">").replace("&lt;", "<")  # decode HTML
    t = re.sub(r"[ \t]+", " ", t)         # collapse spaces
    return t.strip()

Emoji, punctuation, casing, and misspellings are deliberately preserved — those carry your slang signal, exactly as the proposal commits. Only identity (usernames, mention handles) and boilerplate (URLs, HTML entities) get stripped.

## Step 3 Cleaning function setup


In [11]:
def clean_frame(d, kind):
    # Anonymise, filter, and normalise one subreddit's posts or comments.
    d = d.copy()
    d["uid"] = d.author.map(hash_user)    # username -> pseudonymous ID

    # Posts and comments store their text in different fields:
    if kind == "posts":
        d["text"] = (d.title.fillna("") + "\n\n" + d.selftext.fillna("")).str.strip()  # title + body
        if "controversiality" not in d:
            d["controversiality"] = 0     # posts lack this field; pad so schema matches
    else:
        d["text"] = d.body.fillna("")     # comment body
        if "num_comments" not in d:
            d["num_comments"] = 0         # comments lack this field; pad so schema matches

    d["text"] = d.text.map(clean_text)    # normalise (see clean_text)

    # Filters in order; log records how many rows each one removes (for methodology)
    log = {}
    n = len(d)
    d = d[d.uid.notna()];                                log["no author/bot"] = n - len(d); n = len(d)  # drop bots/deleted
    d = d[~d.text.isin(["[deleted]", "[removed]", ""])]; log["removed/empty"]  = n - len(d); n = len(d)  # drop removed/empty
    d = d[d.text.str.split().str.len() >= 5];            log["under 5 words"]  = n - len(d); n = len(d)  # drop too-short
    d = d.drop_duplicates(subset=["uid", "text"]);       log["dupe text"]      = n - len(d)              # drop repeats

    # Keep only columns downstream stages use (skips any missing ones safely)
    keep = ["id","uid","subreddit","created_utc","text","score",
            "kind","permalink","num_comments","controversiality"]
    d = d[[c for c in keep if c in d.columns]]
    return d, log   # cleaned frame + attrition log

### Running the clean function on all the files that were capped in step 01

In [12]:
files = sorted(glob.glob(f"{PARTS}/capped_*.parquet"))
print(len(files), "files to clean\n")

audit = []
for p in files:
    name = os.path.basename(p).replace("capped_", "").replace(".parquet", "")
    sub = re.sub(r"^r_", "", name).rsplit("_", 1)[0]   # r_bangtan_posts -> bangtan
    kind = "posts" if name.endswith("_posts") else "comments"
    dest = CLEAN_DAPT if sub in DAPT_ONLY else CLEAN
    out = f"{dest}/{name}.parquet"
    if os.path.exists(out):
        print("skip", name); continue

    d = pd.read_parquet(p)
    d["kind"] = kind
    d, log = clean_frame(d, kind)
    d.to_parquet(out, compression="zstd")

    row = {"sub": sub, "kind": kind, "kept": len(d), "dest": os.path.basename(dest)}
    row.update(log)
    audit.append(row)
    print(f"{name:45s} kept {len(d):>7} {log}")
    del d

audit = pd.DataFrame(audit)
audit.to_csv("data/interim/clean_audit.csv", index=False)

38 files to clean

r_AKB48_comments                              kept   92719 {'no author/bot': 6475, 'removed/empty': 0, 'under 5 words': 12632, 'dupe text': 463}
r_AKB48_posts                                 kept   17967 {'no author/bot': 2422, 'removed/empty': 0, 'under 5 words': 1515, 'dupe text': 132}
r_StrayKids_comments                          kept  235665 {'no author/bot': 14495, 'removed/empty': 0, 'under 5 words': 44309, 'dupe text': 5479}
r_StrayKids_posts                             kept   41916 {'no author/bot': 6565, 'removed/empty': 1, 'under 5 words': 7781, 'dupe text': 832}
r_TWICEsnark_comments                         kept    5368 {'no author/bot': 352, 'removed/empty': 0, 'under 5 words': 456, 'dupe text': 285}
r_TWICEsnark_posts                            kept     323 {'no author/bot': 10, 'removed/empty': 0, 'under 5 words': 33, 'dupe text': 7}
r_bangtan_comments                            kept  246717 {'no author/bot': 16286, 'removed/empty': 5, 'under 5 words': 

### Verification step 

In [14]:
# --- Verify cleaned output ---------------------------------------------------
# Two folders, but note the roles:
#   clean/      -> feature-eligible subs (used for features, clustering, AND DAPT)
#   clean_dapt/ -> popculture only: excluded from features/clustering, DAPT-only
# DAPT trains on BOTH folders combined. The split only gates the feature matrix.

print("FEATURE-ELIGIBLE + DAPT (clean/):")
feat_tot = 0
for p in sorted(glob.glob(f"{CLEAN}/*.parquet")):
    n = len(pd.read_parquet(p, columns=["id"]))
    feat_tot += n
print("  total kept:", feat_tot)

print("\nDAPT-ONLY, excluded from features (clean_dapt/):")
dapt_only_tot = 0
for p in sorted(glob.glob(f"{CLEAN_DAPT}/*.parquet")):
    n = len(pd.read_parquet(p, columns=["id"]))
    dapt_only_tot += n
    print("  ", os.path.basename(p), n)

# The full DAPT corpus is both folders together:
print("\nCORPUS TOTALS:")
print("  feature matrix (clean/ only):", feat_tot)
print("  full DAPT corpus (both):     ", feat_tot + dapt_only_tot)

print("\nattrition summary:")
print(audit.groupby("dest")[["kept"]].sum())

FEATURE-ELIGIBLE + DAPT (clean/):
  total kept: 2777991

DAPT-ONLY, excluded from features (clean_dapt/):
   r_popculture_comments.parquet 246925
   r_popculture_posts.parquet 19912

CORPUS TOTALS:
  feature matrix (clean/ only): 2777991
  full DAPT corpus (both):      3044828

attrition summary:
               kept
dest               
clean       2777991
clean_dapt   266837


Saving attribution attribute

In [15]:
audit.to_csv("data/interim/clean_audit.csv", index=False)